# LangGraph + MCP — Playground Notebook

A tiny, self-contained version of the big project, meant for **learning the
LangGraph primitives in isolation** before reading `app/graph.py`.

What this notebook builds, top to bottom:

1. **Imports & setup** — libraries + load API keys from `.env`
2. **State** — the shared dict every node reads/writes
3. **MCP tools** — connect to remote tool servers (browserless + tavily)
4. **Local tools** — a screenshot tool and a video-download tool
5. **Tool list + LLM** — bind every tool to the model
6. **Build the graph** — `profiler_agent` ⇄ `tools` loop, compiled with a checkpointer
7. **Run it** — a small Gradio chat drives the graph
8. **Inspect** — look at saved state and state history

At the very bottom, **section 11** demos the 5 core concepts one by one:
checkpointing, interrupt → resume, human-in-the-loop routing, branching,
and replay safety.

> Run the cells in order. Each needs the one above it.


## 1. Imports & setup

Pull in LangGraph, LangChain and helpers. `add_messages` is a *reducer*: when
a node returns `{"messages": [...]}`, LangGraph **appends** to the list
instead of overwriting it. `MemorySaver` is an in-memory checkpointer (state
kept in RAM only — fine for a notebook).


In [ ]:
from typing import Annotated
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from dotenv import load_dotenv
from IPython.display import display, Image
import gradio as gr
from langchain_openai import ChatOpenAI
from pydantic import BaseModel
from typing import Annotated, TypedDict
import random
from langgraph.checkpoint.memory import MemorySaver


In [ ]:
load_dotenv(override=True)  # Load environment variables from .env file

## 2. State — the shared memory

`State` is the single dict passed through every node. Here it only holds the
chat `messages`. `Annotated[list, add_messages]` tells LangGraph to merge new
messages onto the existing list. The real project's state is much bigger
(profile result, pinned competitors, research queue, …) but the idea is the
same.


In [ ]:
class State(TypedDict):
    messages: Annotated[list, add_messages]

## 3. MCP tools — remote tool servers

MCP (Model Context Protocol) lets the agent call tools that live on a remote
server. `MultiServerMCPClient` connects to two of them:

- **browserless** — headless-browser scraping / screenshots / crawling
- **tavily** — web search, extract, crawl, research

`await mcp_client.get_tools()` returns them as normal LangChain tools. The
`print` loop just lists what we got.


In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv
from langchain_mcp_adapters.client import MultiServerMCPClient

load_dotenv()

BROWSERLESS_TOKEN = os.getenv("BROWSERLESS_TOKEN")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")

PROJECT_DIR = str(Path.cwd().resolve())


mcp_client = MultiServerMCPClient(
    {
        # Browser automation / scraping
        "browserless": {
            "transport": "http",
            "url": "https://mcp.browserless.io/mcp",
            "headers": {
                "Authorization": f"Bearer {BROWSERLESS_TOKEN}"
            },
        },

        # Search / extract / crawl / research
        "tavily": {
            "transport": "http",
            "url": f"https://mcp.tavily.com/mcp/?tavilyApiKey={TAVILY_API_KEY}",
        },

        # Local project files
        # "filesystem": {
        #     "transport": "stdio",
        #     "command": "npx",
        #     "args": [
        #         "-y",
        #         "@modelcontextprotocol/server-filesystem",
        #         PROJECT_DIR,
        #     ],
        # },
    }
)

mcp_tools = await mcp_client.get_tools()

for t in mcp_tools:
    print(t.name)

## 4. Local tools

Two plain Python tools defined with the `@tool` decorator, so the LLM can
call them like any MCP tool:

- `download_video(url)` — grabs a video with `yt_dlp` into `./videos/`
- `browserless_screenshot(url)` — saves a PNG into `./screenshots/`

A `@tool` function's **docstring is the description the LLM sees**, so it
matters.


In [ ]:
from pathlib import Path
from datetime import datetime
from langchain_core.tools import tool
import re
import yt_dlp


VIDEO_DIR = Path.cwd() / "videos"
VIDEO_DIR.mkdir(parents=True, exist_ok=True)


@tool
def download_video(
    url: str,
    filename: str = "",
) -> str:
    """
    Download a video from a supported webpage or direct video URL.

    Args:
        url: URL containing the video.
        filename: Optional output filename without extension.

    Returns:
        Path to the downloaded video.
    """

    if not filename:
        filename = f"video_{datetime.now():%Y%m%d_%H%M%S}"

    filename = re.sub(
        r"[^a-zA-Z0-9_-]",
        "_",
        filename
    )

    output_template = str(
        VIDEO_DIR / f"{filename}.%(ext)s"
    )

    options = {
        "outtmpl": output_template,

        # Prefer MP4-compatible streams when available
        "format": "bestvideo+bestaudio/best",

        "merge_output_format": "mp4",

        "noplaylist": True,
        "quiet": True,
    }

    with yt_dlp.YoutubeDL(options) as ydl:
        info = ydl.extract_info(
            url,
            download=True
        )

        # Find resulting path
        requested = info.get("requested_downloads")

        if requested:
            path = requested[0].get("filepath")
        else:
            path = ydl.prepare_filename(info)

    return f"Video saved to: {Path(path).resolve()}"


print("Video download tool ready")
print("Folder:", VIDEO_DIR.resolve())

from pathlib import Path
from datetime import datetime
from langchain_core.tools import tool
import re
import requests

# Saves to: <your project>/screenshots/
SCREENSHOT_DIR = Path.cwd() / "screenshots"
SCREENSHOT_DIR.mkdir(parents=True, exist_ok=True)

@tool
def browserless_screenshot(
    url: str,
    filename: str = "",
    full_page: bool = True,
) -> str:
    """
    Take a screenshot of a webpage and save it locally.

    Use this when you need a visual screenshot of a webpage.

    Args:
        url: Full webpage URL.
        filename: Optional filename without extension.
        full_page: Capture the entire webpage if True.

    Returns:
        Path to the saved PNG file.
    """

    # Generate safe filename
    if not filename:
        filename = f"screenshot_{datetime.now():%Y%m%d_%H%M%S}"

    filename = re.sub(r"[^a-zA-Z0-9_-]", "_", filename)

    output_path = SCREENSHOT_DIR / f"{filename}.png"

    endpoint = (
        "https://production-sfo.browserless.io/screenshot"
        f"?token={BROWSERLESS_TOKEN}"
    )

    response = requests.post(
        endpoint,
        headers={
            "Cache-Control": "no-cache",
            "Content-Type": "application/json",
        },
        json={
            "url": url,
            "options": {
                "fullPage": full_page,
                "type": "png",
            },
        },
        timeout=90,
    )

    response.raise_for_status()

    output_path.write_bytes(response.content)

    return f"Screenshot saved to: {output_path.resolve()}"


print("Screenshot tool ready")
print("Folder:", SCREENSHOT_DIR.resolve())

## 5. Tool list + LLM

Combine MCP tools + local tools into one `tools` list, create the chat model,
and `llm.bind_tools(tools)` so the model knows it is allowed to call them.
`llm_with_tools` is what the agent node uses.


In [ ]:
from langchain_core.tools import Tool
from langgraph.prebuilt import ToolNode
from langgraph.prebuilt import tools_condition
import json

tools = [
    *mcp_tools,
    browserless_screenshot, download_video,
]

In [ ]:
from pathlib import Path

# .env lives in backend/, not the repo root where this notebook sits
load_dotenv(Path.cwd() / "backend" / ".env", override=True)

llm = ChatOpenAI(
    model_name="openai/gpt-5.6-luna",
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ["OPENROUTER_API_KEY"],
)
llm_with_tools = llm.bind_tools(tools)


## 6. Build the graph

The classic agent loop as a graph:

```
START → profiler_agent → (tools_condition) → tools → profiler_agent → ...
                                            ↘ __end__
```

- **`profiler_agent`** — calls the LLM; if the reply contains `tool_calls`
  it returns them, otherwise it's the final answer.
- **`tools`** — `ToolNode` runs whichever tool the agent asked for and feeds
  the result back.
- **`tools_condition`** — a prebuilt router: "did the last message ask for a
  tool?" → go to `tools`, else → `END`.
- **`compile(checkpointer=memory)`** — every step is now saved, keyed by a
  `thread_id` (see next section).

The last line renders the graph as a picture.


In [ ]:
PROFILER_PROMPT = """
Profile the company from its website.

Rules:
- Be efficient.
- Start with the homepage.
- Use at most 3 rounds of tool calls.
- Do not crawl every page.
- Do not call the same tool repeatedly for information already available.
- Prefer one search/extraction call covering multiple relevant pages.
- If some information cannot be found, return "Unknown".
- Once you have enough information to produce the requested profile,
  STOP using tools and return the final result.
"""

async def profiler_agent(state: State):
    print("\n=== PROFILER ===")

    messages = [
        SystemMessage(content=PROFILER_PROMPT),
        *state["messages"],
    ]

    response = await llm_with_tools.ainvoke(messages)

    if response.tool_calls:
        for call in response.tool_calls:
            print(f"TOOL: {call['name']}")
            print(f"ARGS: {call['args']}")

    return {"messages": [response]}




graph_builder = StateGraph(State)
graph_builder.add_node("profiler_agent", profiler_agent)
graph_builder.add_node("tools", ToolNode(tools=tools))

graph_builder.add_edge(START, "profiler_agent")

graph_builder.add_conditional_edges(
    "profiler_agent",
    tools_condition,
    {
        "tools": "tools",
        "__end__": END,
    },
)
graph_builder.add_edge("tools", "profiler_agent")

memory = MemorySaver()  # Initialize the memory saver for checkpointing
graph = graph_builder.compile(checkpointer=memory)
display(Image(graph.get_graph().draw_mermaid_png()))

## 7. Run it — a small Gradio chat

Every app launch picks a fresh `thread_id`, wrapped in a `config` dict. That
`config` is passed to **every** graph call — it's how the checkpointer knows
which conversation this is.

`graph.astream(state, config, stream_mode="updates")` runs the graph and
yields one event per node as it finishes. At the end we read the final
answer back with `graph.aget_state(config)`.


In [ ]:
import uuid
import gradio as gr
from langchain_core.messages import HumanMessage, SystemMessage

# New thread every app launch
thread_id = str(uuid.uuid4())

config = {
    "configurable": {
        "thread_id": thread_id
    }
}


async def chat(user_input: str, history):

    initial_state = {
        "messages": [
            HumanMessage(content=user_input)
        ]
    }

    async for event in graph.astream(
        initial_state,
        config=config,
        stream_mode="updates",
    ):
        print("\nEVENT:", event)

    state = await graph.aget_state(config)

    return state.values["messages"][-1].content


print("Thread ID:", thread_id)

demo = gr.ChatInterface(
    fn=chat,
    title="LangGraph Chatbot",
    description="A chatbot powered by LangGraph."
)

demo.launch()

## 8. Inspect the saved state

- `graph.get_state(config)` — the **current** snapshot for this thread
  (values, and `next` = which node would run next; empty `next` means done).
- `graph.get_state_history(config)` — **every** checkpoint for the thread,
  newest first. This is the list `branch_from_checkpoint()` walks in the real
  project.


In [ ]:
graph.get_state(config)

In [ ]:
list(graph.get_state_history(config))

# 11. The 5 core concepts, one at a time

The cells below build a **tiny toy graph** (no LLM, no tools) so each concept
is easy to see on its own. Same primitives the real project uses in
`app/graph.py`.

Toy graph shape:

```
START → plan → review → (decision) → plan      ("redo"  → loop back)
                                    → finalize  ("done"  → advance)
finalize → END
```

- `plan` bumps a counter and appends to a log
- `review` **pauses** with `interrupt()` and waits for a human answer
- the human's answer is **parsed** and routes the graph (loop or advance)


In [ ]:
# --- Build the toy graph ---------------------------------------------------
import sqlite3, operator
from pathlib import Path
from typing import Annotated, TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.types import interrupt, Command
from langgraph.checkpoint.sqlite import SqliteSaver


class ToyState(TypedDict):
    count: int
    log: Annotated[list, operator.add]   # reducer: appends
    decision: str


def plan(state: ToyState):
    # NOTE: this runs (and prints) every time the node is entered, including
    # on replay. Keep real side effects out of nodes like this. (see 11.5)
    n = state["count"] + 1
    print(f"  [plan] run #{n}")
    return {"count": n, "log": [f"planned #{n}"]}


def review(state: ToyState):
    # interrupt() freezes the graph here and hands this payload to the caller.
    answer = interrupt({"ask": "approve this plan?", "options": ["done", "redo"]})
    print(f"  [review] human said: {answer!r}")
    return {"decision": answer.strip().lower()}


def finalize(state: ToyState):
    print("  [finalize] done")
    return {"log": ["finalized"]}


def route_after_review(state: ToyState) -> str:
    # parsed human command -> which edge to take
    return "plan" if state["decision"] == "redo" else "finalize"


builder = StateGraph(ToyState)
builder.add_node("plan", plan)
builder.add_node("review", review)
builder.add_node("finalize", finalize)
builder.add_edge(START, "plan")
builder.add_edge("plan", "review")
builder.add_conditional_edges("review", route_after_review,
                              {"plan": "plan", "finalize": "finalize"})
builder.add_edge("finalize", END)

# One SQLite file on disk (in this notebook's folder) -> survives a
# kernel/process restart. Deleted first so the demos below are repeatable.
DB_PATH = "toy_checkpoints.db"
try:
    conn.close()  # close a connection left open by a previous run of this cell
except NameError:
    pass
Path(DB_PATH).unlink(missing_ok=True)
conn = sqlite3.connect(DB_PATH, check_same_thread=False)
toy = builder.compile(checkpointer=SqliteSaver(conn))
print("toy graph ready")


## 11.0 Visualize the toy graph

Same trick as section 6: `toy.get_graph().draw_mermaid_png()` renders the
`plan → review → (decision) → plan | finalize` shape as a picture (via
`mermaid.ink`). If that network call fails, fall back to the ASCII drawing.


In [ ]:
from IPython.display import display, Image

try:
    display(Image(toy.get_graph().draw_mermaid_png()))
except Exception as e:
    print(f"(mermaid render failed: {e})\n")
    print(toy.get_graph().draw_ascii())


## 11.1 Checkpointing

**Idea:** every step is written to a SQLite file, keyed by a `thread_id`. If
the process dies, you rebuild the graph from the *same file* and the thread
picks up exactly where it stopped.

- Real project: `AsyncSqliteSaver` → `data/checkpoints.db`; `data/runs.csv`
  is a human-friendly index of `thread_id`s (`main.py --history`).
- Below: we start a run, it pauses at `review`, then we throw the graph
  object away and rebuild it — the thread state is still there.


In [ ]:
cfg = {"configurable": {"thread_id": "demo-1"}}

print("first run:")
for ev in toy.stream({"count": 0, "log": []}, cfg):
    pass

snap = toy.get_state(cfg)
print("paused at:", snap.next, "| count =", snap.values["count"])

# Simulate a process restart: brand-new graph object, same .db file.
toy2 = builder.compile(checkpointer=SqliteSaver(
    sqlite3.connect(DB_PATH, check_same_thread=False)))
print("after 'restart', thread still knows:", toy2.get_state(cfg).values)


## 11.2 Interruption → continue

**Idea:** `interrupt(payload)` inside a node stops the graph and bubbles
`payload` out to whoever called `.stream()`. You resume by calling the graph
again with `Command(resume=<answer>)` — the node re-runs and `interrupt()`
now *returns* that answer.

- Real project: `interrupt()` lives in `competitor_review` and
  `research_scope_review`; `run_interrupt_loop` catches the `__interrupt__`
  event, prints the menu, reads one line, resumes via `Command(resume=...)`.


In [ ]:
# We're still paused at 'review' on thread demo-1. See the interrupt payload:
snap = toy.get_state(cfg)
print("interrupt payload:", snap.interrupts[0].value)

# Resume with a human answer:
for ev in toy.stream(Command(resume="done"), cfg):
    pass

snap = toy.get_state(cfg)
print("next:", snap.next, "(empty = finished)")
print("log:", snap.values["log"])


## 11.3 Human-in-the-loop → routing

**Idea:** the resumed answer isn't just stored — it is *parsed into a
command* and a **conditional edge** uses that command to decide the next
node: loop back for another round, or move forward.

- Real project: `parse_command()` turns the typed line into
  `pin` / `add` / `done` (competitors) or `select` / `add` / `done`
  (research scope). `competitor_review` / `research_scope_review` then route:
  `add` → regenerate, `pin`/`select` → stay in the review loop, `done` →
  finalize.
- Below: answering `"redo"` sends the graph back to `plan`; `"done"` sends it
  to `finalize`. Watch `[plan] run #2` appear on the redo.


In [ ]:
cfg2 = {"configurable": {"thread_id": "demo-routing"}}

# start -> pauses at review
for ev in toy.stream({"count": 0, "log": []}, cfg2):
    pass

# answer "redo" -> route_after_review returns "plan" -> loop back
print('answering "redo":')
for ev in toy.stream(Command(resume="redo"), cfg2):
    pass

# now paused again; answer "done" -> route to finalize
print('answering "done":')
for ev in toy.stream(Command(resume="done"), cfg2):
    pass

print("final log:", toy.get_state(cfg2).values["log"])


## 11.4 Branching

**Idea:** `get_state_history(config)` gives every checkpoint of a thread. Pick
one, take its `.values`, and start a **new** `thread_id` seeded with those
values — then let it run forward on its own. The old thread is untouched.

- Real project: `branch_from_checkpoint()` walks the history, keeps the
  latest checkpoint that wrote each significant node
  (`finalize_profile`, `finalize_competitors`, …), you pick one, and its
  `state.values` seed a fresh `runs_db` row / thread. Because each node
  checks its per-stage CSV cache by `company_id` first, the branch
  fast-forwards through already-done stages.
- Below: we pick the checkpoint right after the first `plan` (here we match
  on `.next`; the real project matches on `metadata["writes"]`) and seed a
  new thread from it.


In [ ]:
# Every checkpoint of the demo-routing thread (newest first).
# `.next` = which node would run next from that checkpoint.
history = list(toy.get_state_history(cfg2))
for h in history:
    print(f"step={h.metadata['step']:>2}  next={h.next}  count={h.values.get('count')}")

# Pick the checkpoint just after the FIRST 'plan' (about to run 'review', count == 1):
source = next(h for h in history
              if h.next == ("review",) and h.values["count"] == 1)
print("\npicked step:", source.metadata["step"], "->", source.values)

# Seed a BRAND-NEW thread with that checkpoint's values, then let it run forward.
branch_cfg = {"configurable": {"thread_id": "demo-branch"}}
seed = {"count": source.values["count"], "log": list(source.values["log"])}
for ev in toy.stream(seed, branch_cfg):
    pass
for ev in toy.stream(Command(resume="done"), branch_cfg):
    pass

print("\nbranch   log:", toy.get_state(branch_cfg).values["log"])
print("original log:", toy.get_state(cfg2).values["log"], "(untouched)")


## 11.5 Replay safety

**Idea:** when a thread resumes, LangGraph **re-runs the current node from its
start** to rebuild in-memory context (the code before `interrupt()` runs
again). So:

- A node that calls `interrupt()` must be **safe to run twice**: only build
  the review payload / read state — no writing to CSV, no API calls, no
  charging a card. That is why in the real project **only** `competitor_review`
  and `research_scope_review` call `interrupt()`, and they are
  presentation-only.
- Nodes that *do* side effects (`competitor_discovery` search,
  `finalize_competitors` CSV writes, `research`) sit **before or after** the
  interrupt node and never call `interrupt()` themselves — so a resume replay
  can't double-fire them.
- In our toy: `plan` prints on every entry, including replays. Fine for a
  `print`; would be a bug if `plan` inserted a database row.

Rule of thumb: **interrupt only in a node that would be harmless to run
again.**
